# Picotron 26M Parameter Model Pre-Training Pipeline
This notebook configures, tokenizes, shards, packs, and trains a **26M parameter LLaMA model** on **1B tokens** across **3 epochs** combining Fineweb-Edu and Cosmopedia data sources.

## Step 1: Environment Setup

In [ ]:
# Set base directory and clone fresh repository
%cd /kaggle/working

# Clean up previous clone attempts if they exist
!rm -rf picotron
!pip uninstall -y picotron
!git clone https://github.com/Syntropy-AI-Labs/picotron.git

# Install in editable mode
%cd picotron
!pip install -e .

## Step 2: Write 26M Parameter Model Config Profile
Generates `config_26M_foundation.yaml` configuring a ~26M model structure tailored for 1B training tokens.

In [ ]:
%%writefile config_26M_foundation.yaml
model:
  vocab_size: 50257                # GPT-2 tokenizer dictionary size
  hidden_size: 256                 # Latent hidden dimensions
  num_hidden_layers: 16            # Layer depth
  num_attention_heads: 8           # Head count
  num_key_value_heads: 4           # Grouped Query Attention ratio 2:1
  intermediate_size: 1024          # Feed-Forward expansion
  max_position_embeddings: 512     # Maximum context window
  norm_type: "rms"
  activation_type: "silu"
  rms_norm_eps: 0.000005
  bias: false

parallel:
  dp_size: 1
  zero_stage: 1                    # Enable Stage 1 Optimizer Sharding

data:
  dataset_path: "data/foundation_tokens.bin"
  sequence_length: 512
  micro_batch_size: 16
  num_workers: 8

train:
  learning_rate: 0.001
  min_learning_rate: 0.0001
  weight_decay: 0.1
  max_steps: 19531                 # 1B tokens / (16 * 512) = ~122k steps. For 3 epochs: ~366k steps.
                                   # Adjusting max_steps to represent target tokens context.
  warmup_steps: 2000
  grad_accum_steps: 4
  seed: 42
  compile: true
  use_cuda_graphs: false
  mixed_precision: "bf16"
  save_checkpoint: true
  checkpoint_dir: "checkpoints/26M_foundation"

## Step 3: Run High-Performance Async Preprocessing
Leverages the `picotron-data` preprocessing engine to dynamically stream, clean, tokenize, and sequence-pack fineweb-edu and cosmopedia datasets utilizing all CPU cores.

In [ ]:
# Create data directory
!mkdir -p data

# Clear existing output file if it exists
!rm -f data/foundation_tokens.bin

# Download, clean, tokenize, shard, pack and cache
# Streams 500M tokens from fineweb-edu and 500M tokens from cosmopedia
!picotron-data \
  --dataset HuggingFaceFW/fineweb-edu \
  --config sample-10BT \
  --tokenizer gpt2 \
  --output_dir data/ \
  --seq_len 512 \
  --num_workers 8 \
  --limit_tokens 500000000

!picotron-data \
  --dataset HuggingFaceTB/cosmopedia \
  --config auto-selected \
  --tokenizer gpt2 \
  --output_dir data/ \
  --seq_len 512 \
  --num_workers 8 \
  --limit_tokens 500000000

## Step 4: Run Foundation Pre-Training
Launches the pre-training execution loop using the custom 26M configuration.

In [ ]:
# Force clear stale imports cache from Jupyter kernel memory
import sys
for mod in list(sys.modules.keys()):
    if "picotron" in mod:
        del sys.modules[mod]

# Run Picotron Pre-Training
!picotron-train config_26M_foundation.yaml